# 🎯 Task 3: BestBuy Deal Finder - Gradio App

**Mục tiêu:** Tạo Gradio web app cho phép user:
1. Nhập keyword tìm kiếm (ví dụ: "laptop", "Smart TV")
2. Xem danh sách deals với **giá sale** và **giá ước lượng**
3. Sắp xếp theo **discount** (ước lượng - sale)
4. Nhận **push notification** khi tìm thấy deal tốt

**Workflow:**
```
User Input → BestBuySearchAgent → filter_sale_urls → scrape_bestbuy_products 
          → BestBuyScannerAgent → EnsembleAgent → Opportunities → Gradio Table
```

In [1]:
# Cell 1: Setup - Đảm bảo working directory là segment4
import os
import sys
import logging

# Chuyển working directory về segment4 (quan trọng!)
os.chdir('/home/hieu0606sunny/price2026wsl/tech2ai/segment4')
sys.path.insert(0, '/home/hieu0606sunny/price2026wsl/tech2ai/segment4')

print(f"Working directory: {os.getcwd()}")

# Setup logging
logging.basicConfig(level=logging.INFO)
root = logging.getLogger()
root.setLevel(logging.INFO)

from dotenv import load_dotenv
load_dotenv(override=True)

print("✅ Setup complete!")

Working directory: /home/hieu0606sunny/price2026wsl/tech2ai/segment4
✅ Setup complete!


In [2]:
# Cell 2: Test imports từ các modules đã có
from typing import List
import chromadb
import gradio as gr
import queue
import threading
import time
import asyncio

# Import từ bestbuy modules (Task 1 & 2)
from price_agents.bestbuy_deals import (
    ScrapedBestBuyDeal,
    is_on_sale,
    filter_sale_urls,
    scrape_bestbuy_products
)
from price_agents.bestbuy_scanner_agent import (
    BestBuySearchAgent,
    BestBuyScannerAgent
)

# Import từ existing modules
from price_agents.ensemble_agent import EnsembleAgent
from price_agents.messaging_agent import MessagingAgent
from price_agents.deals import Deal, DealSelection, Opportunity
from log_utils import reformat  # For color formatting

print("✅ All imports successful!")

INFO:datasets:PyTorch version 2.9.0 available.


✅ All imports successful!


In [3]:
# Cell 3: Khởi tạo ChromaDB và EnsembleAgent
# (Đây là bước tốn thời gian nhất, chỉ cần chạy 1 lần)

print("Initializing ChromaDB...")
DB_PATH = "products_vectorstore"
client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection('products')
print(f"ChromaDB collection: {collection.name}")
print(f"Number of documents: {collection.count()}")

print("\nInitializing EnsembleAgent...")
ensemble = EnsembleAgent(collection)
print("✅ EnsembleAgent ready!")

print("\nInitializing MessagingAgent...")
messenger = MessagingAgent()
print("✅ MessagingAgent ready!")

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Initializing ChromaDB...


INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Specialist Agent] Specialist Agent is ready
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI


ChromaDB collection: products
Number of documents: 800000

Initializing EnsembleAgent...


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using cuda
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready
INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and GPT


✅ EnsembleAgent ready!

Initializing MessagingAgent...
✅ MessagingAgent ready!


## 📦 Step-by-Step Pipeline Functions

Trước khi tạo Gradio UI, hãy tạo các hàm pipeline riêng biệt để dễ test.

In [4]:
# Cell 5: Pipeline Functions

def search_bestbuy(keyword: str, max_urls: int = 10) -> List[str]:
    """
    Step 1: Tìm URLs từ BestBuy bằng Brave Search.
    
    Args:
        keyword: Từ khóa tìm kiếm (vd: "Smart TV")
        max_urls: Số URLs tối đa
    
    Returns:
        List[str]: Danh sách URLs
    """
    search_agent = BestBuySearchAgent()
    urls = search_agent.search(keyword, max_urls=max_urls)
    return urls


def filter_sales(urls: List[str]) -> List[str]:
    """
    Step 2: Lọc chỉ giữ URLs có sản phẩm đang khuyến mãi.
    
    Args:
        urls: Danh sách URLs từ search
    
    Returns:
        List[str]: URLs có sản phẩm đang sale
    """
    return filter_sale_urls(urls)


async def scrape_products(urls: List[str]) -> List[ScrapedBestBuyDeal]:
    """
    Step 3: Cào dữ liệu từ BestBuy với Playwright.
    
    Args:
        urls: Danh sách URLs sản phẩm sale
    
    Returns:
        List[ScrapedBestBuyDeal]: Dữ liệu sản phẩm
    """
    return await scrape_bestbuy_products(urls, headless=False)


def select_top_deals(scraped_deals: List[ScrapedBestBuyDeal]) -> DealSelection:
    """
    Step 4: Chọn top 5 deals bằng GPT-5-mini.
    
    Args:
        scraped_deals: Dữ liệu đã cào
    
    Returns:
        DealSelection: Top 5 deals
    """
    scanner = BestBuyScannerAgent()
    return scanner.scan(scraped_deals)


def estimate_prices(deal_selection: DealSelection, ensemble_agent: EnsembleAgent) -> List[Opportunity]:
    """
    Step 5: Dự đoán giá và tính discount.
    
    Args:
        deal_selection: Top deals đã chọn
        ensemble_agent: EnsembleAgent đã khởi tạo
    
    Returns:
        List[Opportunity]: Danh sách cơ hội, sorted by discount
    """
    opportunities = []
    
    for deal in deal_selection.deals:
        estimate = ensemble_agent.price(deal.product_description)
        discount = estimate - deal.price
        opportunity = Opportunity(
            deal=deal,
            estimate=estimate,
            discount=discount
        )
        opportunities.append(opportunity)
    
    # Sort by discount descending
    opportunities.sort(key=lambda x: x.discount, reverse=True)
    return opportunities


print("✅ Pipeline functions defined!")

✅ Pipeline functions defined!


In [5]:
# Cell 6: Test Pipeline với sample URLs (không cần Brave API)
# Dùng URLs từ test trước để verify pipeline hoạt động

sample_urls = [
    "https://www.bestbuy.com/product/samsung-55-class-u7900-series-uhd-4k-smart-tizen-tv-2025/J3ZYG2V5FW",
    "https://www.bestbuy.com/product/westinghouse-24-class-smart-tv-hd-xumo-tv-with-voice-remote-flat-screen-led-television/J3LL89C69K",
]

print(f"Testing pipeline with {len(sample_urls)} sample URLs...")
print("Step 2: Filtering sale items...")
sale_urls = filter_sales(sample_urls)
print(f"  → {len(sale_urls)} sale URLs")

Testing pipeline with 2 sample URLs...
Step 2: Filtering sale items...


INFO:price_agents.bestbuy_deals:[1/2] SALE
INFO:price_agents.bestbuy_deals:[2/2] SALE
INFO:price_agents.bestbuy_deals:Filtered 2 URLs → 2 sale URLs


  → 2 sale URLs


In [6]:
# Cell 7: Test Scraping (Browser sẽ mở)

print("Step 3: Scraping with Playwright...")
scraped_deals = await scrape_products(sale_urls)
print(f"  → Scraped {len(scraped_deals)} products")

# Hiển thị kết quả
for i, deal in enumerate(scraped_deals, 1):
    print(f"\n  [{i}] {deal.title[:50]}...")
    print(f"      Price: ${deal.price}")

Step 3: Scraping with Playwright...


INFO:price_agents.bestbuy_deals:[1/2] Scraping: https://www.bestbuy.com/product/samsung-55-class-u7900-serie...
INFO:price_agents.bestbuy_deals:  ✓ <Samsung - 55" Class U7900 Series UHD 4K Smart Tize... | $279.99>
INFO:price_agents.bestbuy_deals:[2/2] Scraping: https://www.bestbuy.com/product/westinghouse-24-class-smart-...
INFO:price_agents.bestbuy_deals:  ✓ <Westinghouse - 24” Class Smart TV, HD Xumo TV with... | $79.99>
INFO:price_agents.bestbuy_deals:Successfully scraped 2/2 products


  → Scraped 2 products

  [1] Samsung - 55" Class U7900 Series UHD 4K Smart Tize...
      Price: $279.99

  [2] Westinghouse - 24” Class Smart TV, HD Xumo TV with...
      Price: $79.99


In [7]:
# Cell 8: Test Select Top Deals

print("Step 4: Selecting top deals with GPT-5-mini...")
deal_selection = select_top_deals(scraped_deals)
print(f"  → Selected {len(deal_selection.deals)} deals")

# Hiển thị
for i, deal in enumerate(deal_selection.deals, 1):
    print(f"\n  [{i}] {deal.product_description[:60]}...")
    print(f"      Price: ${deal.price}")

INFO:root:[BestBuy Scanner Agent] BestBuy Scanner Agent is initializing
INFO:root:[BestBuy Scanner Agent] BestBuy Scanner Agent is ready
INFO:root:[BestBuy Scanner Agent] Calling gpt-5-mini with 2 deals...


Step 4: Selecting top deals with GPT-5-mini...


INFO:root:[BestBuy Scanner Agent] Selected 2 deals


  → Selected 2 deals

  [1] A 55-inch UHD 4K LED television powered by Samsung's Crystal...
      Price: $279.99

  [2] A compact 24-inch LED HD smart TV offering a 720p resolution...
      Price: $79.99


In [8]:
# Cell 9: Test Estimate Prices

print("Step 5: Estimating prices with EnsembleAgent...")
opportunities = estimate_prices(deal_selection, ensemble)
print(f"  → Estimated {len(opportunities)} opportunities")

# Hiển thị kết quả
print("\n" + "=" * 60)
print("RESULTS (sorted by discount):")
print("=" * 60)

for i, opp in enumerate(opportunities, 1):
    discount_pct = (opp.discount / opp.estimate * 100) if opp.estimate > 0 else 0
    print(f"\n🏷️ #{i}: {opp.deal.product_description[:50]}...")
    print(f"   💰 Sale: ${opp.deal.price:.2f}")
    print(f"   📊 Estimate: ${opp.estimate:.2f}")
    print(f"   🎯 Discount: ${opp.discount:.2f} ({discount_pct:.1f}%)")

INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
20:00:52 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


Step 5: Estimating prices with EnsembleAgent...


20:00:53 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $648.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $589.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $422.46
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $578.25
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
20:01:30 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
20:01:31 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Sp

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $109.99
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $143.07
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $124.30


  → Estimated 2 opportunities

RESULTS (sorted by discount):

🏷️ #1: A 55-inch UHD 4K LED television powered by Samsung...
   💰 Sale: $279.99
   📊 Estimate: $578.25
   🎯 Discount: $298.26 (51.6%)

🏷️ #2: A compact 24-inch LED HD smart TV offering a 720p ...
   💰 Sale: $79.99
   📊 Estimate: $124.30
   🎯 Discount: $44.31 (35.6%)


---

## 🖥️ Gradio UI

Bây giờ đã verify pipeline hoạt động, hãy tạo Gradio UI.

In [9]:
# Cell 11: Gradio Helper Functions + Real-time Logging

# Global variables
current_opportunities = []
log_queue = None  # Will be set when search starts


class QueueHandler(logging.Handler):
    """Custom logging handler that puts logs into a queue for Gradio."""
    def __init__(self, log_queue):
        super().__init__()
        self.log_queue = log_queue

    def emit(self, record):
        self.log_queue.put(self.format(record))


def setup_logging(q):
    """Setup logging to capture all logs into queue."""
    global log_queue
    log_queue = q
    
    handler = QueueHandler(q)
    formatter = logging.Formatter(
        "[%(asctime)s] %(message)s",
        datefmt="%H:%M:%S",
    )
    handler.setFormatter(formatter)
    
    # Add handler to root logger
    logger = logging.getLogger()
    # Remove existing handlers to avoid duplicates
    for h in logger.handlers[:]:
        if isinstance(h, QueueHandler):
            logger.removeHandler(h)
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)


def html_for_logs(log_data: List[str]) -> str:
    """Convert log data to HTML for display."""
    # Keep last 20 logs
    recent_logs = log_data[-20:]
    # Format each log with color
    formatted = [reformat(log) for log in recent_logs]
    output = '<br>'.join(formatted)
    return f"""
    <div style="height: 350px; overflow-y: auto; border: 1px solid #444; 
                background-color: #1a1a2e; padding: 10px; font-family: monospace; 
                font-size: 12px; border-radius: 8px;">
    {output}
    </div>
    """


def opportunities_to_table(opportunities: List[Opportunity]) -> list:
    """
    Convert opportunities thành data cho Gradio Dataframe.
    
    Returns:
        List of rows: [Product, Sale $, Estimate $, Discount $, Discount %, URL]
    """
    rows = []
    for opp in opportunities:
        discount_pct = (opp.discount / opp.estimate * 100) if opp.estimate > 0 else 0
        status = "🔥" if opp.discount > 200 else ("✅" if opp.discount > 100 else ("👍" if opp.discount > 0 else "❌"))
        rows.append([
            opp.deal.product_description[:60] + "...",
            f"${opp.deal.price:.2f}",
            f"${opp.estimate:.2f}",
            f"${opp.discount:.2f}",
            f"{discount_pct:.1f}% {status}",
            opp.deal.url
        ])
    return rows


print("✅ Gradio helpers + logging defined!")

✅ Gradio helpers + logging defined!


In [14]:
# Cell 12: Main Gradio Search Function with Real-time Logging
# Sử dụng threading + queue để hiển thị logs liên tục

def do_search_pipeline(keyword: str, max_urls: int, result_queue: queue.Queue):
    """
    Worker function chạy trong thread riêng.
    Kết quả sẽ được đưa vào result_queue.
    """
    global current_opportunities
    
    try:
        # Step 1: Search
        logging.info(f"🔍 [Step 1/5] Searching for '{keyword}' on BestBuy...")
        urls = search_bestbuy(keyword, max_urls)
        if not urls:
            result_queue.put(([], "❌ No products found. Try a different keyword."))
            return
        logging.info(f"✅ Found {len(urls)} product URLs")
        
        # Step 2: Filter sales
        logging.info(f"🏷️ [Step 2/5] Filtering sale items from {len(urls)} URLs...")
        sale_urls = filter_sales(urls)
        if not sale_urls:
            result_queue.put(([], "❌ No sale items found. Try a different keyword."))
            return
        logging.info(f"✅ Found {len(sale_urls)} products on SALE")
        
        # Step 3: Scrape (async)
        logging.info(f"📦 [Step 3/5] Scraping {len(sale_urls)} products with Playwright...")
        
        # Handle async in sync context
        try:
            loop = asyncio.get_event_loop()
        except RuntimeError:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
        
        scraped = loop.run_until_complete(scrape_products(sale_urls))
        if not scraped:
            result_queue.put(([], "❌ Could not scrape product details."))
            return
        logging.info(f"✅ Scraped {len(scraped)} products successfully")
        
        # Step 4: Select top deals
        logging.info(f"🤖 [Step 4/5] Selecting top 5 deals with GPT-5-mini...")
        deal_selection = select_top_deals(scraped)
        if not deal_selection or not deal_selection.deals:
            result_queue.put(([], "❌ Could not select deals."))
            return
        logging.info(f"✅ Selected {len(deal_selection.deals)} best deals")
        
        # Step 5: Estimate prices
        logging.info(f"💰 [Step 5/5] Estimating prices with EnsembleAgent (3 models)...")
        opportunities = estimate_prices(deal_selection, ensemble)
        current_opportunities = opportunities  # Save for push notification
        logging.info(f"✅ Estimated {len(opportunities)} opportunities")
        
        if opportunities:
            best = opportunities[0]
            logging.info(f"🏆 BEST DEAL: ${best.deal.price:.2f} → Est: ${best.estimate:.2f} = Discount ${best.discount:.2f}")
        
        # Return results
        table = opportunities_to_table(opportunities)
        status = f"✅ Found {len(opportunities)} deals! Best discount: ${opportunities[0].discount:.2f}" if opportunities else "No deals found."
        
        result_queue.put((table, status))
        
    except Exception as e:
        import traceback
        traceback.print_exc()
        result_queue.put(([], f"❌ Error: {str(e)}"))


def run_search_with_logging(keyword: str, max_urls: int, initial_log_data: List[str]):
    """
    Generator function cho Gradio.
    Yield logs liên tục trong khi pipeline chạy.
    """
    if not keyword or len(keyword.strip()) < 2:
        yield [], "❌ Please enter a keyword (at least 2 characters)", html_for_logs(["❌ Invalid keyword"])
        return
    
    # Setup
    log_q = queue.Queue()
    result_q = queue.Queue()
    setup_logging(log_q)
    
    log_data = initial_log_data.copy() if initial_log_data else []
    log_data.append(f"🚀 Starting search for: {keyword}")
    
    # Start worker thread
    thread = threading.Thread(
        target=do_search_pipeline,
        args=(keyword, int(max_urls), result_q)
    )
    thread.start()
    
    # Yield updates while thread is running
    final_result = None
    while thread.is_alive() or not log_q.empty() or final_result is None:
        # Check for new logs
        try:
            while True:
                message = log_q.get_nowait()
                log_data.append(message)
        except queue.Empty:
            pass
        
        # Check for result
        try:
            final_result = result_q.get_nowait()
        except queue.Empty:
            pass
        
        # Yield current state
        if final_result:
            table, status = final_result
            yield table, status, html_for_logs(log_data)
        else:
            yield [], "⏳ Processing...", html_for_logs(log_data)
        
        time.sleep(0.1)
    
    # Final yield
    if final_result:
        table, status = final_result
        log_data.append(f"✅ Pipeline completed!")
        yield table, status, html_for_logs(log_data)


print("✅ Main search function with real-time logging defined!")

✅ Main search function with real-time logging defined!


In [15]:
# Cell 13: Push Notification Function

def send_push_notification(selected_index: int) -> str:
    """
    Gửi push notification cho deal được chọn.
    
    Args:
        selected_index: Index của deal trong bảng (0-based)
    
    Returns:
        Status message
    """
    global current_opportunities
    
    print(f"[DEBUG] send_push_notification called with index: {selected_index}")
    print(f"[DEBUG] current_opportunities count: {len(current_opportunities) if current_opportunities else 0}")
    
    if not current_opportunities:
        msg = "❌ No deals available. Search first!"
        print(f"[DEBUG] Returning: {msg}")
        return msg
    
    idx = int(selected_index)
    if idx < 0 or idx >= len(current_opportunities):
        msg = f"❌ Invalid selection. Choose 0-{len(current_opportunities)-1}"
        print(f"[DEBUG] Returning: {msg}")
        return msg
    
    opp = current_opportunities[idx]
    print(f"[DEBUG] Sending notification for: {opp.deal.product_description[:30]}...")
    
    try:
        messenger.notify(
            description=opp.deal.product_description[:200],
            deal_price=opp.deal.price,
            estimated_true_value=opp.estimate,
            url=opp.deal.url
        )
        msg = f"✅ Sent! Deal #{idx}: {opp.deal.product_description[:40]}... (${opp.deal.price:.2f})"
        print(f"[DEBUG] Success: {msg}")
        return msg
    except Exception as e:
        msg = f"❌ Failed: {str(e)}"
        print(f"[DEBUG] Error: {msg}")
        return msg


print("✅ Push notification function defined!")

✅ Push notification function defined!


In [19]:
# Cell 14: Gradio UI Definition với Real-time Logs

def create_gradio_app():
    """Create and return Gradio app with real-time logging."""
    
    with gr.Blocks(
        title="BestBuy Deal Finder",
        theme=gr.themes.Soft(),
        fill_width=True,
    ) as app:
        
        # State for logs
        log_data_state = gr.State([])
        
        # Header
        gr.Markdown("""
        # 🔍 BestBuy Deal Finder
        
        Tìm kiếm sản phẩm trên BestBuy và phát hiện deals tốt nhất bằng AI!
        
        **Pipeline:** 1️⃣ Search → 2️⃣ Filter Sales → 3️⃣ Scrape → 4️⃣ Select Top 5 → 5️⃣ Estimate Prices
        """)
        
        # Search Section
        with gr.Row():
            keyword_input = gr.Textbox(
                label="🔎 Keyword",
                placeholder="Enter product keyword (e.g., Smart TV, laptop, headphones)",
                scale=4
            )
            max_urls_input = gr.Number(
                label="Max URLs",
                value=10,
                minimum=1,
                maximum=50,
                precision=0,
                scale=1
            )
            search_btn = gr.Button("🔍 Search", variant="primary", scale=1)
        
        # Status
        status_text = gr.Textbox(
            label="Status",
            interactive=False,
            value="Ready to search..."
        )
        
        # Main content: Results + Logs side by side
        with gr.Row():
            # Left: Results Table
            with gr.Column(scale=3):
                results_table = gr.Dataframe(
                    headers=["Product", "Sale $", "Estimate $", "Discount $", "Discount %", "URL"],
                    label="🏆 Deal Results (sorted by discount)",
                    wrap=True,
                    column_widths=[4, 1, 1, 1, 1, 2],
                    max_height=350,
                )
            
            # Right: Real-time Logs
            with gr.Column(scale=2):
                gr.Markdown("### 📋 Pipeline Logs")
                logs_html = gr.HTML(
                    value='<div style="height: 350px; background-color: #1a1a2e; border-radius: 8px; padding: 10px; font-family: monospace; color: #87CEEB;">Ready to search...</div>'
                )
        
        # Push Notification Section
        with gr.Row():
            deal_index = gr.Number(
                label="Deal # to notify (0 = best deal)",
                value=0,
                minimum=0,
                precision=0,
                scale=1
            )
            push_btn = gr.Button("📱 Send Push Notification", variant="secondary", scale=1)
        
        # Notification status (separate row for better visibility)
        push_status = gr.Textbox(
            label="📱 Notification Status",
            interactive=False,
            placeholder="Click 'Send Push Notification' to send...",
            lines=2,
        )
        
        # Event handlers
        search_btn.click(
            fn=run_search_with_logging,
            inputs=[keyword_input, max_urls_input, log_data_state],
            outputs=[results_table, status_text, logs_html]
        )
        
        push_btn.click(
            fn=send_push_notification,
            inputs=[deal_index],
            outputs=[push_status],
            api_name="send_notification"
        )
        
        # Footer
        gr.Markdown("""
        ---
        **Legend:** 🔥 Discount > $200 | ✅ Discount > $100 | 👍 Positive discount | ❌ Overpriced
        
        **Notes:** Browser window will open during scraping (Playwright). Push notifications require Pushover setup.
        """)
    
    return app


print("✅ Gradio UI with real-time logs defined!")

✅ Gradio UI with real-time logs defined!


In [20]:
# Cell 15: Launch Gradio App
# ⚠️ Chỉ chạy cell này khi đã test xong các cells trước!

app = create_gradio_app()
app.launch(share=False, inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


INFO:root:🔍 [Step 1/5] Searching for 'Phone' on BestBuy...
INFO:root:[BestBuy Search Agent] BestBuy Search Agent is initializing
INFO:root:[BestBuy Search Agent] BestBuy Search Agent is ready
INFO:root:[BestBuy Search Agent] Searching for: Phone
INFO:root:[BestBuy Search Agent] Found 15 product URLs
INFO:root:✅ Found 15 product URLs
INFO:root:🏷️ [Step 2/5] Filtering sale items from 15 URLs...
INFO:price_agents.bestbuy_deals:[1/15] SALE
INFO:price_agents.bestbuy_deals:[2/15] Skip
INFO:price_agents.bestbuy_deals:[3/15] Skip
INFO:price_agents.bestbuy_deals:[4/15] SALE
INFO:price_agents.bestbuy_deals:[5/15] Skip
INFO:price_agents.bestbuy_deals:[6/15] Skip
INFO:price_agents.bestbuy_deals:[7/15] Skip
INFO:price_agents.bestbuy_deals:[8/15] Skip
INFO:price_agents.bestbuy_deals:[9/15] Skip
INFO:price_agents.bestbuy_deals:[10/15] Skip
INFO:price_agents.bestbuy_deals:[11/15] Skip
INFO:price_agents.bestbuy_deals:[12/15] SALE
INFO:price_agents.bestbuy_deals:[13/15] Skip
INFO:price_agents.bestbuy_de

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $799.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $266.75
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $695.77
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
20:41:50 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
20:41:50 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Sp

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $89.99
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $95.61
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $87.55
INFO:root:✅ Estimated 4 opportunities
INFO:root:🏆 BEST DEAL: $179.99 → Est: $321.46 = Discount $141.47
INFO:root:[Messaging Agent] Messaging Agent is using GPT to craft the message
20:42:31 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= gpt-5-nano; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-5-nano; provider = openai


[DEBUG] send_push_notification called with index: 0
[DEBUG] current_opportunities count: 4
[DEBUG] Sending notification for: The Samsung Galaxy A17 5G is a...


20:42:56 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed


[DEBUG] Success: ✅ Sent! Deal #0: The Samsung Galaxy A17 5G is a midrange ... ($179.99)
